In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("27-jobs-stages-tasks")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]

## Section 1 — Model overview (comments)

In [ ]:
print("=" * 70)
print("SECTION 1: Jobs / Stages / Tasks — Mental Model")
print("=" * 70)

# JOB:   triggered by an action (count, show, collect, write, save)
# STAGE: a sequence of narrow transformations; bounded by a shuffle (wide transformation)
# TASK:  one unit of work = one partition × one stage; runs on one executor core
#
# Key insight: number of tasks per stage = number of partitions going INTO that stage
#
# Narrow transformation  → no shuffle → same stage  (map, filter, withColumn)
# Wide   transformation  → shuffle    → new stage   (groupBy, join, repartition)
print("(See inline comments above for model overview)")

## Section 2 — Trigger a job with count() and observe in UI

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: Single-action Jobs (browse http://localhost:4040/jobs)")
print("=" * 70)

print("Action 1: emp.count()")
print(emp.count())  # Job 1 in Spark UI

print("\nAction 2: sal.count()")
print(sal.count())  # Job 2 in Spark UI

# Comment: Browse http://localhost:4040/jobs to see these two jobs listed.

## Section 3 — Multi-stage job (groupBy triggers shuffle = 2 stages)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: groupBy → shuffle → 2 stages (browse /stages)")
print("=" * 70)

print("\nAction 3: groupBy+count (triggers shuffle → 2 stages)")
emp.groupBy("dept_id").count().show()
# Stage 1: read + partial aggregate (narrow)
# Shuffle:  Exchange (sort or hash partition on dept_id)
# Stage 2:  final aggregate (narrow)
# Visit http://localhost:4040/stages to see stage details

## Section 4 — Join job (stages and tasks)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Join Job — multiple stages from two shuffles")
print("=" * 70)

print("\nAction 4: join employees + salary_history (autoBroadcast disabled)")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "4")
emp.join(sal, "emp_id").count()
# Expected: 3–4 stages (scan emp, scan sal, shuffle both sides, join + count)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
print("Done — check Spark UI Stages tab for Exchange nodes.")

## Section 5 — Task slots and parallelism

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Task Slots and Parallelism")
print("=" * 70)

sc = spark.sparkContext
print(f"\nDefault parallelism: {sc.defaultParallelism}")
print(f"Available cores:     {sc.defaultParallelism}")  # local[*] = number of CPU cores
# In local[*]:  tasks run sequentially on the driver thread pool.
# In a cluster: each executor core handles one task concurrently.
# spark.sql.shuffle.partitions controls tasks in the post-shuffle stage (default 200).
print(f"shuffle.partitions:  {spark.conf.get('spark.sql.shuffle.partitions')}")

## Section 6 — Skewed stage (Engineering dept dominates)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Skewed Stage Demo (Engineering = 14/41 employees)")
print("=" * 70)

spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
emp.repartition(4, "dept_id").join(sal.repartition(4, "emp_id"), "emp_id") \
   .groupBy("dept_id").agg(F.sum("salary_after")).show()
# In Spark UI Stages: one task may have duration much longer than others = skew indicator.
# Engineering (dept_id=1) has ~34% of employees → its partition is the largest.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
print("Skew demo done — in Stages tab, compare task durations per partition.")

## Section 7 — Skipped stages (cached data)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Skipped Stages (cache reuse)")
print("=" * 70)

emp_cached = emp.filter(F.col("status") == "Active").cache()
emp_cached.count()  # Job N:   scans data, materialises cache
emp_cached.count()  # Job N+1: should show skipped stages (cached partition reused)
# In Spark UI: skipped stages are shown in green / strikethrough.
print("Two count() jobs fired — second should show skipped stages in UI.")
emp_cached.unpersist()

## Section 8 — Task metrics interpretation (comment block)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: Task Metrics Interpretation (comments only)")
print("=" * 70)

# In Spark UI  Stages > click a stage > Task Metrics table:
#
#   Duration      — wall clock time per task; wide variation = skew
#   Shuffle Read  — bytes read from shuffle (output of previous stage)
#   Shuffle Write — bytes written to shuffle (input to next stage)
#   GC Time       — Java garbage collection; high GC % = memory pressure
#   Input Size    — bytes read from source (files / in-memory)
#   Spill (Disk)  — data that didn't fit in executor memory → local disk = slow
#
# Healthy stage:  Duration variance low, Spill = 0, GC Time < 10% of Duration
# Skewed stage:   Max Duration >> Median Duration (one slow task)
# Memory issue:   GC Time > 10% Duration OR Spill (Disk) > 0
print("(See inline comments above for metric interpretations)")

stop_and_wait(spark)